In [3]:

from google.colab import files
import pandas as pd
import os


uploaded = files.upload()


print("Files uploaded:")
print(os.listdir())


file_name = [f for f in os.listdir() if f.endswith(".xlsx")][0]


df = pd.read_excel(file_name)

print("Dataset loaded successfully!")
print("File name:", file_name)
print("Shape:", df.shape)

Saving midterm-regresi-dataset (1).xlsx to midterm-regresi-dataset (1).xlsx
Files uploaded:
['.config', 'midterm-regresi-dataset (1).xlsx', 'sample_data']
Dataset loaded successfully!
File name: midterm-regresi-dataset (1).xlsx
Shape: (515344, 91)


In [5]:


print("=== DATASET SHAPE ===")
print(df.shape)

print("\n=== FIRST 5 ROWS ===")
print(df.head())

print("\n=== DATASET INFO ===")
df.info()

print("\n=== MISSING VALUES (TOP 20) ===")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing.head(20))

print("\n=== DUPLICATE ROWS ===")
print(df.duplicated().sum())

print("\n=== BASIC STATISTICS ===")
print(df.describe())

=== DATASET SHAPE ===
(515344, 91)

=== FIRST 5 ROWS ===
    2001.00000   49.94357     21.47114     73.07750     8.74861     \
0         2001     48.73215     18.42930     70.32679     12.94636   
1         2001     50.95714     31.85602     55.81851     13.41693   
2         2001     48.24750     -1.89837     36.29772      2.58776   
3         2001     50.97020     42.20998     67.09964      8.46791   
4         2001     50.54767      0.31568     92.35066     22.38696   

   -17.40628    -13.09905    -25.01202    -12.23257     7.83089     ...  \
0    -10.32437    -24.83777      8.76630     -0.92019     18.76548  ...   
1     -6.57898    -18.54940     -3.27872     -2.35035     16.07017  ...   
2      0.97170    -26.21683      5.05097    -10.34124      3.55005  ...   
3    -15.85279    -16.81409    -12.48207     -9.37636     12.63699  ...   
4    -25.51870    -19.04928     20.67345     -5.19943      3.63566  ...   

    13.01620    -54.40548     58.99367     15.37344     1.11144     \
0

In [6]:


import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_cols = df.shape[1]
df.columns = ["target_year"] + [f"feature_{i}" for i in range(1, num_cols)]

print("Columns renamed successfully!")

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

X = df.drop("target_year", axis=1)
y = df["target_year"]

imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)


X = pd.DataFrame(X_imputed, columns=X.columns)

Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

X = X.clip(lower=lower_bound, upper=upper_bound, axis=1)

print("Outlier handling completed!")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed!")

Columns renamed successfully!
Shape after removing duplicates: (515130, 91)
Outlier handling completed!
Train shape: (412104, 90)
Test shape: (103026, 90)
Scaling completed!


In [7]:

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

results = []

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    results.append([name, mae, mse, rmse, r2])

    print(f"\n=== {name} ===")
    print("MAE:", mae)
    print("MSE:", mse)
    print("RMSE:", rmse)
    print("R²:", r2)


lr = LinearRegression()
evaluate_model("Linear Regression", lr, X_train_scaled, X_test_scaled, y_train, y_test)


rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
evaluate_model("Random Forest", rf, X_train, X_test, y_train, y_test)


xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)
evaluate_model("XGBoost", xgb, X_train, X_test, y_train, y_test)


results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "MSE", "RMSE", "R2"]
)

print("\n=== MODEL COMPARISON ===")
print(results_df)


=== Linear Regression ===
MAE: 6.70536397779128
MSE: 88.4935725984736
RMSE: 9.40710224237377
R²: 0.25452079097821634


KeyboardInterrupt: 

In [ ]:


!pip install optuna

import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

def objective(trial):

    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))

    return rmse

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

print("=== BEST PARAMETERS ===")
print(study.best_params)

print("\n=== BEST RMSE ===")
print(study.best_value)

In [ ]:
best_params = study.best_params

best_rf = RandomForestRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train, y_train)

best_preds = best_rf.predict(X_test)

mae = mean_absolute_error(y_test, best_preds)
mse = mean_squared_error(y_test, best_preds)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, best_preds)

print("=== BEST RANDOM FOREST RESULTS ===")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R²:", r2)

In [ ]:

!pip install lime

from lime.lime_tabular import LimeTabularExplainer


explainer = LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_train.columns,
    mode="regression"
)


exp = explainer.explain_instance(
    X_test.iloc[0],
    best_rf.predict,
    num_features=10
)


exp.show_in_notebook(show_table=True)

In [ ]:
!pip install mlflow

import mlflow
import mlflow.sklearn

mlflow.set_experiment("Regression Midterm Project")

with mlflow.start_run():


    mlflow.log_params(best_params)


    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("MSE", mse)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)


    mlflow.sklearn.log_model(best_rf, "best_random_forest_model")

print("MLflow tracking completed!")